In [3]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = PROJECT_DIR / "processed"
PLOTS_DIR = PROJECT_DIR / "Plots"

PROCESSED_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

DB_PATH = DATA_DIR / "PlasticEnz.db"

print("Project:", PROJECT_DIR)
print("Database:", DB_PATH)
print("Database exists:", DB_PATH.exists())

Project: d:\Machine-Learning\Paper-1\project
Database: d:\Machine-Learning\Paper-1\project\data\PlasticEnz.db
Database exists: True


In [4]:
import sqlite3
conn = sqlite3.connect(DB_PATH)

print("Connected to PlasticEnz database")

Connected to PlasticEnz database


In [5]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,sqlite_sequence
1,Plastic
2,Enzymes
3,Reaction
4,Microorganisms
5,Databases
6,Study
7,Main_copy
8,Sequences


In [6]:
main = pd.read_sql_query("SELECT * FROM Main_copy", conn)
enzymes = pd.read_sql_query("SELECT * FROM Enzymes", conn)
sequences = pd.read_sql_query("SELECT * FROM Sequences", conn)
plastics = pd.read_sql_query("SELECT * FROM Plastic", conn)

In [7]:
print("Main:", main.shape)
print("Enzymes:", enzymes.shape)
print("Sequences:", sequences.shape)
print("Plastics:", plastics.shape)

Main: (279, 13)
Enzymes: (62, 8)
Sequences: (272, 2)
Plastics: (13, 3)


In [8]:
print("MAIN")
print(main.columns.tolist())

print("\nENZYMES")
print(enzymes.columns.tolist())

print("\nSEQUENCES")
print(sequences.columns.tolist())

print("\nPLASTICS")
print(plastics.columns.tolist())

MAIN
['Genus', 'Species', 'Strain', 'Enzyme', 'Optimal_conditions', 'Main_Pk', 'Database_id', 'Sequences_id', 'Study_id', 'Enzymes_id', 'Microorganism_id', 'Reaction_id', 'Plastic_id']

ENZYMES
['Location', 'Gene', 'Type', 'Superfamily', 'Operon', 'Gene_Cluster', 'Catalytic domain', 'Enzymes_id']

SEQUENCES
['SEQUENCE', 'Seq_Pk']

PLASTICS
['Plastic_id', 'PLASTIC', 'TYPE']


In [9]:
feature_cols = [
    "Type",
    "Superfamily",
    "Catalytic domain",
    "Location"
]

for col in feature_cols:
    print(f"\n===== {col} =====")
    print(enzymes[col].value_counts(dropna=False))


===== Type =====
Type
Acting on ester bonds                                                           47
Acting on paired donors, with incorporation or reduction of molecular oxygen     7
Acting on peptide bonds                                                          4
Acting on diphenols and related substances as donors;                            1
Acting on the aldehyde or oxo group of donors                                    1
Acting on the CH-OH group of donors                                              1
Forming carbon-sulfur bonds                                                      1
Name: count, dtype: int64

===== Superfamily =====
Superfamily
Hydrolases         51
Oxidoreductases    10
Ligases             1
Name: count, dtype: int64

===== Catalytic domain =====
Catalytic domain
          29
NaN       27
Type 1     2
Type 2     1
Type 3     1
Type 4     1
Type 5     1
Name: count, dtype: int64

===== Location =====
Location
Extracellular          52
Cell inner membrane 

In [10]:
feature_summary = pd.DataFrame({
    "non_missing": enzymes[feature_cols].notna().sum(),
    "missing": enzymes[feature_cols].isna().sum(),
    "unique_values": enzymes[feature_cols].nunique(dropna=True)
})

feature_summary

,non_missing,missing,unique_values
Type,62,0,7
Superfamily,62,0,3
Catalytic domain,35,27,6
Location,62,0,5


In [11]:
enzymes[feature_cols].drop_duplicates().sort_values(feature_cols)

,Type,Superfamily,Catalytic domain,Location
20,Acting on diphenols and related substances as ...,Oxidoreductases,,Extracellular
5,Acting on ester bonds,Hydrolases,,Cell outer membrane
3,Acting on ester bonds,Hydrolases,,Extracellular
0,Acting on ester bonds,Hydrolases,Type 1,Extracellular
2,Acting on ester bonds,Hydrolases,Type 2,Extracellular
43,Acting on ester bonds,Hydrolases,Type 3,Extracellular
44,Acting on ester bonds,Hydrolases,Type 4,Extracellular
45,Acting on ester bonds,Hydrolases,Type 5,Extracellular
32,Acting on ester bonds,Hydrolases,NaN,Extracellular
21,"Acting on paired donors, with incorporation or...",Oxidoreductases,,Cell inner membrane


In [12]:
enzymes["Catalytic domain"].value_counts(dropna=False)

Catalytic domain
          29
NaN       27
Type 1     2
Type 2     1
Type 3     1
Type 4     1
Type 5     1
Name: count, dtype: int64

In [13]:
print("MAIN")
display(main.head())

print("ENZYMES")
display(enzymes.head())

print("SEQUENCES")
display(sequences.head())

print("PLASTICS")
display(plastics.head())

MAIN


,Genus,Species,Strain,Enzyme,Optimal_conditions,Main_Pk,Database_id,Sequences_id,Study_id,Enzymes_id,Microorganism_id,Reaction_id,Plastic_id
0,Streptomyces,hygroscopicus,ascomyceticus,PHB depolymerase,he enzyme was found to be a monomer with a mol...,1,1.0,1.0,1.0,1.0,1.0,1.0,3.0
1,Bacillus,megaterium,N-18-25-9,PHB depolymerase,The maximum activity was observed at pH 9.0 at...,2,2.0,2.0,2.0,2.0,2.0,1.0,3.0
2,Pseudomonas,lemoignei,,PHB depolymerase,NaN,3,3.0,3.0,3.0,2.0,3.0,1.0,3.0
3,Alcaligenes,faecalis,,PHB depolymerase,NaN,4,4.0,4.0,4.0,2.0,4.0,1.0,3.0
4,Ralstonia,pickettii,T1,PHB depolymerase,NaN,5,5.0,5.0,5.0,2.0,5.0,1.0,3.0


ENZYMES


,Location,Gene,Type,Superfamily,Operon,Gene_Cluster,Catalytic domain,Enzymes_id
0,Extracellular,fkbU (PhaZ),Acting on ester bonds,Hydrolases,,FK520,Type 1,1
1,Extracellular,PhaZ,Acting on ester bonds,Hydrolases,pha operon,,Type 1,2
2,Extracellular,PhaZ,Acting on ester bonds,Hydrolases,pha operon,,Type 2,3
3,Extracellular,AMED_7492,Acting on ester bonds,Hydrolases,,,,4
4,Extracellular,ISF6_4831,Acting on ester bonds,Hydrolases,,,,5


SEQUENCES


,SEQUENCE,Seq_Pk
0,MQPPPFRGILTPLFPLSSSPPVGSLSRPGRRGVLTRLVAVVALVLG...,1
1,MKRLFIAGMIFILFLSLGAVSSSAAGSFTSKTYNGRTYKLYVPSSY...,2
2,MLAKQIKKANSRSTLLRKSLLFAAPIILAVSSSSVYALTQVSNF\n...,3
3,MTSRPMRSLVIAFLTLVAAAAPALAGAGAWQNNLSLGGFNKVHLYT...,4
4,MVRRLWRRIAGWLAACVAILCAFPLHAATAGPGAWSSQQTWAADSV...,5


PLASTICS


,Plastic_id,PLASTIC,TYPE
0,1,PET,Fossil fuel-based
1,2,PE,Fossil fuel-based
2,3,PHB,Renewable resources
3,4,PLA,Renewable resources
4,5,PCL,Fossil fuel-based


In [14]:
print("Main columns:")
print(main.columns.tolist())

print("\nEnzyme columns:")
print(enzymes.columns.tolist())

print("\nSequence columns:")
print(sequences.columns.tolist())

print("\nPlastic columns:")
print(plastics.columns.tolist())

Main columns:
['Genus', 'Species', 'Strain', 'Enzyme', 'Optimal_conditions', 'Main_Pk', 'Database_id', 'Sequences_id', 'Study_id', 'Enzymes_id', 'Microorganism_id', 'Reaction_id', 'Plastic_id']

Enzyme columns:
['Location', 'Gene', 'Type', 'Superfamily', 'Operon', 'Gene_Cluster', 'Catalytic domain', 'Enzymes_id']

Sequence columns:
['SEQUENCE', 'Seq_Pk']

Plastic columns:
['Plastic_id', 'PLASTIC', 'TYPE']


In [15]:
model1_raw = (
    main
    .merge(
        enzymes,
        on="Enzymes_id",
        how="left"
    )
    .merge(
        sequences,
        left_on="Sequences_id",
        right_on="Seq_Pk",
        how="left"
    )
    .merge(
        plastics,
        on="Plastic_id",
        how="left"
    )
)

print("Shape:", model1_raw.shape)
display(model1_raw.head())

Shape: (279, 24)


,Genus,Species,Strain,Enzyme,Optimal_conditions,Main_Pk,Database_id,Sequences_id,Study_id,Enzymes_id,...,Gene,Type,Superfamily,Operon,Gene_Cluster,Catalytic domain,SEQUENCE,Seq_Pk,PLASTIC,TYPE
0,Streptomyces,hygroscopicus,ascomyceticus,PHB depolymerase,he enzyme was found to be a monomer with a mol...,1,1.0,1.0,1.0,1.0,...,fkbU (PhaZ),Acting on ester bonds,Hydrolases,,FK520,Type 1,MQPPPFRGILTPLFPLSSSPPVGSLSRPGRRGVLTRLVAVVALVLG...,1.0,PHB,Renewable resources
1,Bacillus,megaterium,N-18-25-9,PHB depolymerase,The maximum activity was observed at pH 9.0 at...,2,2.0,2.0,2.0,2.0,...,PhaZ,Acting on ester bonds,Hydrolases,pha operon,,Type 1,MKRLFIAGMIFILFLSLGAVSSSAAGSFTSKTYNGRTYKLYVPSSY...,2.0,PHB,Renewable resources
2,Pseudomonas,lemoignei,,PHB depolymerase,NaN,3,3.0,3.0,3.0,2.0,...,PhaZ,Acting on ester bonds,Hydrolases,pha operon,,Type 1,MLAKQIKKANSRSTLLRKSLLFAAPIILAVSSSSVYALTQVSNF\n...,3.0,PHB,Renewable resources
3,Alcaligenes,faecalis,,PHB depolymerase,NaN,4,4.0,4.0,4.0,2.0,...,PhaZ,Acting on ester bonds,Hydrolases,pha operon,,Type 1,MTSRPMRSLVIAFLTLVAAAAPALAGAGAWQNNLSLGGFNKVHLYT...,4.0,PHB,Renewable resources
4,Ralstonia,pickettii,T1,PHB depolymerase,NaN,5,5.0,5.0,5.0,2.0,...,PhaZ,Acting on ester bonds,Hydrolases,pha operon,,Type 1,MVRRLWRRIAGWLAACVAILCAFPLHAATAGPGAWSSQQTWAADSV...,5.0,PHB,Renewable resources


In [16]:
print("Missing sequences:", model1_raw["SEQUENCE"].isna().sum())
print("Missing enzymes:", model1_raw["Enzymes_id"].isna().sum())
print("Missing plastics:", model1_raw["PLASTIC"].isna().sum())

print("\nPolymer counts:")
print(model1_raw["PLASTIC"].value_counts(dropna=False))

Missing sequences: 8
Missing enzymes: 7
Missing plastics: 7

Polymer counts:
PLASTIC
PET          74
PHB          55
PCL          46
PBSA         23
PBAT         18
n-alkanes    17
PLA          12
PHA           9
Nylon         9
NaN           7
NR            4
PE            3
MHET          1
PHO           1
Name: count, dtype: int64


In [17]:
missing_rows = model1_raw[
    model1_raw["SEQUENCE"].isna() |
    model1_raw["Enzymes_id"].isna() |
    model1_raw["PLASTIC"].isna()
]

display(
    missing_rows[
        [
            "Main_Pk",
            "Enzymes_id",
            "Sequences_id",
            "Plastic_id",
            "SEQUENCE",
            "PLASTIC"
        ]
    ]
)

,Main_Pk,Enzymes_id,Sequences_id,Plastic_id,SEQUENCE,PLASTIC
94,96,20.0,97.0,9.0,NaN,PBSA
272,279,NaN,NaN,NaN,NaN,NaN
273,280,NaN,NaN,NaN,NaN,NaN
274,281,NaN,NaN,NaN,NaN,NaN
275,282,NaN,NaN,NaN,NaN,NaN
276,283,NaN,NaN,NaN,NaN,NaN
277,284,NaN,NaN,NaN,NaN,NaN
278,285,NaN,NaN,NaN,NaN,NaN


In [18]:
print(missing_rows.columns.tolist())

['Genus', 'Species', 'Strain', 'Enzyme', 'Optimal_conditions', 'Main_Pk', 'Database_id', 'Sequences_id', 'Study_id', 'Enzymes_id', 'Microorganism_id', 'Reaction_id', 'Plastic_id', 'Location', 'Gene', 'Type', 'Superfamily', 'Operon', 'Gene_Cluster', 'Catalytic domain', 'SEQUENCE', 'Seq_Pk', 'PLASTIC', 'TYPE']


In [19]:
display(
    main[main["Main_Pk"] == 94][
        ["Main_Pk", "Enzymes_id", "Sequences_id", "Plastic_id"]
    ]
)

print("\nMatching sequence:")
display(
    sequences[sequences["Seq_Pk"] == 20]
)

,Main_Pk,Enzymes_id,Sequences_id,Plastic_id
92,94,51.0,94.0,9.0



Matching sequence:


,SEQUENCE,Seq_Pk
18,MSLRKSFGLLSATAALVAGLVAAPPAQAAANPYQRGPDPTESLLRA...,20


In [20]:
print("Matching enzyme:")
display(enzymes[enzymes["Enzymes_id"] == 96])

print("\nMatching plastic:")
display(plastics[plastics["Plastic_id"] == 97])

Matching enzyme:


,Location,Gene,Type,Superfamily,Operon,Gene_Cluster,Catalytic domain,Enzymes_id



Matching plastic:


,Plastic_id,PLASTIC,TYPE


In [21]:
print("Matching enzyme:")
display(enzymes[enzymes["Enzymes_id"] == 94])

print("\nMatching sequence:")
display(sequences[sequences["Seq_Pk"] == 51])

print("\nMatching plastic:")
display(plastics[plastics["Plastic_id"] == 94])

Matching enzyme:


,Location,Gene,Type,Superfamily,Operon,Gene_Cluster,Catalytic domain,Enzymes_id



Matching sequence:


,SEQUENCE,Seq_Pk
49,MPFHRFLLAAALASAGMAQAAAPLGQYNIDTGKISVSGLSSGGFMA...,51



Matching plastic:


,Plastic_id,PLASTIC,TYPE


In [22]:
model1_assoc = model1_raw.dropna(
    subset=["Enzymes_id", "Sequences_id", "Plastic_id", "SEQUENCE", "PLASTIC"]
).copy()

print("Original rows:", len(model1_raw))
print("Valid association rows:", len(model1_assoc))
print("Removed rows:", len(model1_raw) - len(model1_assoc))

Original rows: 279
Valid association rows: 271
Removed rows: 8


In [23]:
print(model1_assoc[["Enzymes_id", "Sequences_id", "Plastic_id"]].isna().sum())

Enzymes_id      0
Sequences_id    0
Plastic_id      0
dtype: int64


In [24]:
target_polymers = [
    "PET",
    "PHB",
    "PCL",
    "PBSA",
    "PBAT",
    "PLA",
    "PHA",
    "Nylon"
]

model1_target = model1_assoc[
    model1_assoc["PLASTIC"].isin(target_polymers)
].copy()

print("Rows after target filtering:", len(model1_target))
print("\nTarget counts:")
print(model1_target["PLASTIC"].value_counts())

Rows after target filtering: 245

Target counts:
PLASTIC
PET      74
PHB      55
PCL      46
PBSA     22
PBAT     18
PLA      12
PHA       9
Nylon     9
Name: count, dtype: int64


In [25]:
model1_target["SEQUENCE_NORMALIZED"] = (
    model1_target["SEQUENCE"]
    .astype(str)
    .str.replace(r"\s+", "", regex=True)
    .str.upper()
)

print(model1_target["SEQUENCE_NORMALIZED"].head())

0    MQPPPFRGILTPLFPLSSSPPVGSLSRPGRRGVLTRLVAVVALVLG...
1    MKRLFIAGMIFILFLSLGAVSSSAAGSFTSKTYNGRTYKLYVPSSY...
2    MLAKQIKKANSRSTLLRKSLLFAAPIILAVSSSSVYALTQVSNFGT...
3    MTSRPMRSLVIAFLTLVAAAAPALAGAGAWQNNLSLGGFNKVHLYT...
4    MVRRLWRRIAGWLAACVAILCAFPLHAATAGPGAWSSQQTWAADSV...
Name: SEQUENCE_NORMALIZED, dtype: str


In [26]:
changed = (
    model1_target["SEQUENCE"].astype(str)
    != model1_target["SEQUENCE_NORMALIZED"]
)

print("Sequences changed by normalization:", changed.sum())

Sequences changed by normalization: 98


In [27]:
print("Association rows:", len(model1_target))

print(
    "Unique normalized sequences:",
    model1_target["SEQUENCE_NORMALIZED"].nunique()
)

print(
    "Duplicate association rows:",
    len(model1_target) - model1_target["SEQUENCE_NORMALIZED"].nunique()
)

Association rows: 245
Unique normalized sequences: 188
Duplicate association rows: 57


In [28]:
polymer_per_sequence = (
    model1_target
    .groupby("SEQUENCE_NORMALIZED")["PLASTIC"]
    .nunique()
)

print(polymer_per_sequence.value_counts().sort_index())

PLASTIC
1    156
2     25
3      4
4      2
5      1
Name: count, dtype: int64


In [29]:
protein_sequences = (
    model1_target[
        ["SEQUENCE_NORMALIZED"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

protein_sequences["protein_id"] = [
    f"PE_PROT_{i+1:03d}"
    for i in range(len(protein_sequences))
]

protein_sequences = protein_sequences[
    ["protein_id", "SEQUENCE_NORMALIZED"]
]

print("Unique proteins:", len(protein_sequences))
display(protein_sequences.head())

Unique proteins: 188


,protein_id,SEQUENCE_NORMALIZED
0,PE_PROT_001,MQPPPFRGILTPLFPLSSSPPVGSLSRPGRRGVLTRLVAVVALVLG...
1,PE_PROT_002,MKRLFIAGMIFILFLSLGAVSSSAAGSFTSKTYNGRTYKLYVPSSY...
2,PE_PROT_003,MLAKQIKKANSRSTLLRKSLLFAAPIILAVSSSSVYALTQVSNFGT...
3,PE_PROT_004,MTSRPMRSLVIAFLTLVAAAAPALAGAGAWQNNLSLGGFNKVHLYT...
4,PE_PROT_005,MVRRLWRRIAGWLAACVAILCAFPLHAATAGPGAWSSQQTWAADSV...


In [30]:
import re

valid_aa_pattern = re.compile(r"^[ACDEFGHIKLMNPQRSTVWY]+$")

protein_sequences["valid_sequence"] = (
    protein_sequences["SEQUENCE_NORMALIZED"]
    .apply(lambda x: bool(valid_aa_pattern.fullmatch(x)))
)

print("Valid sequences:", protein_sequences["valid_sequence"].sum())
print("Invalid sequences:", (~protein_sequences["valid_sequence"]).sum())

if (~protein_sequences["valid_sequence"]).sum() > 0:
    display(
        protein_sequences[
            ~protein_sequences["valid_sequence"]
        ]
    )

Valid sequences: 187
Invalid sequences: 1


,protein_id,SEQUENCE_NORMALIZED,valid_sequence
48,PE_PROT_049,MLAPRSLLFCLLLFALPNAFADSRCSERAKTLLLPAKVSCSYKTTW...,False


In [31]:
protein_sequences["sequence_length"] = (
    protein_sequences["SEQUENCE_NORMALIZED"].str.len()
)

print(protein_sequences["sequence_length"].describe())

count    188.000000
mean     342.601064
std      119.782760
min      129.000000
25%      262.000000
50%      303.000000
75%      407.250000
max      914.000000
Name: sequence_length, dtype: float64


In [32]:
display(
    protein_sequences.sort_values("sequence_length")[
        ["protein_id", "sequence_length"]
    ].head(10)
)

,protein_id,sequence_length
143,PE_PROT_144,129
69,PE_PROT_070,181
59,PE_PROT_060,183
58,PE_PROT_059,184
127,PE_PROT_128,186
100,PE_PROT_101,194
171,PE_PROT_172,197
36,PE_PROT_037,201
96,PE_PROT_097,203
64,PE_PROT_065,213


In [33]:
bad_seq = protein_sequences[
    protein_sequences["valid_sequence"] == False
]

for _, row in bad_seq.iterrows():
    print("Protein:", row["protein_id"])
    print("Sequence:", repr(row["SEQUENCE_NORMALIZED"]))
    print("Length:", len(row["SEQUENCE_NORMALIZED"]))
    print(
        "Characters outside standard amino acids:",
        sorted(
            set(row["SEQUENCE_NORMALIZED"]) -
            set("ACDEFGHIKLMNPQRSTVWY")
        )
    )

Protein: PE_PROT_049
Sequence: 'MLAPRSLLFCLLLFALPNAFADSRCSERAKTLLLPAKVSCSYKTTWIDSGLIGARQVIYQAPLGTPPAGGWPVVLIYQGSFFPLNNFTYYSNQPFGGYYEGKLVRALLDSGYAVIAPSAPADLFWQTNIPGLAQAYELTTDYDFLGNVFDAIASGHFGPLNSQRKYATGISSGGYNTSRMAVSFPGQFKALAIQSGSYATCSGPLCVVPDQLPADHXPTYFLHGFVDLTVPWWSMDLYYDRLLQQGIETARYTEPLGGHEWFAASPGKVLAWFNAHP'
Length: 277
Characters outside standard amino acids: ['X']


In [34]:
fasta_path = PROCESSED_DIR / "model1_protein_sequences.fasta"

with open(fasta_path, "w") as f:
    for _, row in protein_sequences.iterrows():
        f.write(f">{row['protein_id']}\n")
        f.write(f"{row['SEQUENCE_NORMALIZED']}\n")

print("FASTA file:", fasta_path)
print("Number of sequences:", len(protein_sequences))

FASTA file: d:\Machine-Learning\Paper-1\project\processed\model1_protein_sequences.fasta
Number of sequences: 188


In [36]:
header_count = fasta_text.count(">PE_PROT_")

print("FASTA headers:", header_count)

FASTA headers: 188


Catalytic-Domain annotation by using InterPro

In [37]:
batch1 = protein_sequences.iloc[:100]
batch2 = protein_sequences.iloc[100:]

fasta1_path = PROCESSED_DIR / "model1_interpro_batch1.fasta"
fasta2_path = PROCESSED_DIR / "model1_interpro_batch2.fasta"

def write_fasta(df, path):
    with open(path, "w") as f:
        for _, row in df.iterrows():
            f.write(f">{row['protein_id']}\n")
            f.write(f"{row['SEQUENCE_NORMALIZED']}\n")

write_fasta(batch1, fasta1_path)
write_fasta(batch2, fasta2_path)

print("Batch 1:", len(batch1))
print("Batch 2:", len(batch2))

Batch 1: 100
Batch 2: 88


Preparation of Target for multi label classifaction

In [39]:
sequence_to_id = dict(
    zip(
        protein_sequences["SEQUENCE_NORMALIZED"],
        protein_sequences["protein_id"]
    )
)

model1_target["protein_id"] = model1_target["SEQUENCE_NORMALIZED"].map(
    sequence_to_id
)

print("Missing protein IDs:", model1_target["protein_id"].isna().sum())

Missing protein IDs: 0


In [40]:
model1_target[["protein_id", "SEQUENCE_NORMALIZED", "PLASTIC"]].head()

,protein_id,SEQUENCE_NORMALIZED,PLASTIC
0,PE_PROT_001,MQPPPFRGILTPLFPLSSSPPVGSLSRPGRRGVLTRLVAVVALVLG...,PHB
1,PE_PROT_002,MKRLFIAGMIFILFLSLGAVSSSAAGSFTSKTYNGRTYKLYVPSSY...,PHB
2,PE_PROT_003,MLAKQIKKANSRSTLLRKSLLFAAPIILAVSSSSVYALTQVSNFGT...,PHB
3,PE_PROT_004,MTSRPMRSLVIAFLTLVAAAAPALAGAGAWQNNLSLGGFNKVHLYT...,PHB
4,PE_PROT_005,MVRRLWRRIAGWLAACVAILCAFPLHAATAGPGAWSSQQTWAADSV...,PHB


In [41]:
target_pairs = (
    model1_target[
        ["protein_id", "PLASTIC"]
    ]
    .drop_duplicates()
    .copy()
)

print("Unique protein-polymer pairs:", len(target_pairs))
print("Unique proteins:", target_pairs["protein_id"].nunique())
print("Unique polymers:", target_pairs["PLASTIC"].nunique())

Unique protein-polymer pairs: 231
Unique proteins: 188
Unique polymers: 8


In [42]:
target_matrix = (
    target_pairs
    .assign(value=1)
    .pivot_table(
        index="protein_id",
        columns="PLASTIC",
        values="value",
        fill_value=0,
        aggfunc="max"
    )
)

target_matrix = target_matrix.reindex(
    index=protein_sequences["protein_id"],
    columns=target_polymers,
    fill_value=0
)

target_matrix = target_matrix.astype(int)

print("Target matrix shape:", target_matrix.shape)
target_matrix.head()

Target matrix shape: (188, 8)


PLASTIC,PET,PHB,PCL,PBSA,PBAT,PLA,PHA,Nylon
protein_id,,,,,,,,
PE_PROT_001,0,1,0,0,0,0,0,0
PE_PROT_002,0,1,0,0,0,0,0,0
PE_PROT_003,0,1,0,0,0,0,0,0
PE_PROT_004,0,1,0,0,0,0,0,0
PE_PROT_005,0,1,0,0,0,0,0,0


In [43]:
label_counts = target_matrix.sum().sort_values(ascending=False)

print(label_counts)

PLASTIC
PET      71
PHB      50
PCL      43
PBSA     20
PBAT     18
PLA      12
Nylon     9
PHA       8
dtype: int64


In [44]:
labels_per_protein = target_matrix.sum(axis=1)

print("Single-label proteins:", (labels_per_protein == 1).sum())
print("Multi-label proteins:", (labels_per_protein > 1).sum())
print("Maximum labels for one protein:", labels_per_protein.max())

Single-label proteins: 156
Multi-label proteins: 32
Maximum labels for one protein: 5


In [45]:
no_label_proteins = target_matrix.index[
    target_matrix.sum(axis=1) == 0
]

print("Proteins with no target label:", len(no_label_proteins))

Proteins with no target label: 0


Feature selection and engineering

In [50]:
model1_features_1A = (
    model1_target[
        ["SEQUENCE_NORMALIZED", "Type", "Superfamily", "Location"]
    ]
    .drop_duplicates(subset=["SEQUENCE_NORMALIZED"])
    .merge(
        protein_sequences,
        on="SEQUENCE_NORMALIZED",
        how="right"
    )
)

model1_features_1A = model1_features_1A[
    ["protein_id", "SEQUENCE_NORMALIZED", "Type", "Superfamily", "Location"]
]

print("Feature table shape:", model1_features_1A.shape)
model1_features_1A

Feature table shape: (188, 5)


,protein_id,SEQUENCE_NORMALIZED,Type,Superfamily,Location
0,PE_PROT_001,MQPPPFRGILTPLFPLSSSPPVGSLSRPGRRGVLTRLVAVVALVLG...,Acting on ester bonds,Hydrolases,Extracellular
1,PE_PROT_002,MKRLFIAGMIFILFLSLGAVSSSAAGSFTSKTYNGRTYKLYVPSSY...,Acting on ester bonds,Hydrolases,Extracellular
2,PE_PROT_003,MLAKQIKKANSRSTLLRKSLLFAAPIILAVSSSSVYALTQVSNFGT...,Acting on ester bonds,Hydrolases,Extracellular
3,PE_PROT_004,MTSRPMRSLVIAFLTLVAAAAPALAGAGAWQNNLSLGGFNKVHLYT...,Acting on ester bonds,Hydrolases,Extracellular
4,PE_PROT_005,MVRRLWRRIAGWLAACVAILCAFPLHAATAGPGAWSSQQTWAADSV...,Acting on ester bonds,Hydrolases,Extracellular
...,...,...,...,...,...
183,PE_PROT_184,MASWQSKILNPVIRALVKRKLLKTHDAAGIRKAFGGVLPPPRGAQF...,Acting on ester bonds,Hydrolases,Extracellular
184,PE_PROT_185,MKKVIVFLVFMLFCTATYAQKVSKVVFFGDSLTDNGNLFRLVELIP...,Acting on ester bonds,Hydrolases,Extracellular
185,PE_PROT_186,MNEAMIRGVLEEAVASGGIAGAVVTVTDANATLFQLAVGRQSVAAP...,Acting on ester bonds,Hydrolases,Extracellular
186,PE_PROT_187,MRTVDGRIGMVERRTRLVSLLAAAALAVSLSGCSATTVLNTLEPRW...,Acting on ester bonds,Hydrolases,Extracellular


In [47]:
print(
    model1_features_1A[
        ["Type", "Superfamily", "Location"]
    ].isna().sum()
)

Type           0
Superfamily    0
Location       0
dtype: int64


In [48]:
for col in ["Type", "Superfamily", "Location"]:
    print(f"\n{col}")
    print(model1_features_1A[col].value_counts(dropna=False))


Type
Type
Acting on ester bonds      183
Acting on peptide bonds      5
Name: count, dtype: int64

Superfamily
Superfamily
Hydrolases    188
Name: count, dtype: int64

Location
Location
Extracellular    188
Name: count, dtype: int64


In [51]:
type_polymer_counts = {}

for polymer in target_polymers:
    mask = target_matrix.loc[
        model1_features_1A["protein_id"],
        polymer
    ].values == 1

    type_polymer_counts[polymer] = (
        model1_features_1A.loc[mask, "Type"]
        .value_counts()
    )

type_polymer_counts = pd.DataFrame(
    type_polymer_counts
).fillna(0).astype(int)

type_polymer_counts

,PET,PHB,PCL,PBSA,PBAT,PLA,PHA,Nylon
Type,,,,,,,,
Acting on ester bonds,71,50,43,20,18,8,8,8
Acting on peptide bonds,0,0,0,0,0,4,0,1


In [53]:
%pip install biopython

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ----------- ---------------------------- 0.8/2.7 MB 4.1 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.7 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 5.0 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [54]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import pandas as pd
import numpy as np

hydrophobic = set("AVILMFWY")
charged = set("DEKR")
aromatic = set("FWY")
polar = set("STNQ")
small = set("AGST")

def sequence_features(seq):
    length = len(seq)
    
    return pd.Series({
        "Sequence_Length": length,
        "Hydrophobic_Fraction": sum(a in hydrophobic for a in seq) / length,
        "Charged_Fraction": sum(a in charged for a in seq) / length,
        "Aromatic_Fraction": sum(a in aromatic for a in seq) / length,
        "Polar_Fraction": sum(a in polar for a in seq) / length,
        "Small_Residue_Fraction": sum(a in small for a in seq) / length,
        "Glycine_Fraction": seq.count("G") / length,
        "Proline_Fraction": seq.count("P") / length,
        "Cysteine_Fraction": seq.count("C") / length,
        "Basic_Fraction": sum(a in "KRH" for a in seq) / length,
        "Acidic_Fraction": sum(a in "DE" for a in seq) / length
    })

sequence_feature_table = protein_sequences.copy()

sequence_feature_table = pd.concat(
    [
        sequence_feature_table,
        sequence_feature_table["SEQUENCE_NORMALIZED"].apply(sequence_features)
    ],
    axis=1
)

sequence_feature_table.head()

,protein_id,SEQUENCE_NORMALIZED,valid_sequence,sequence_length,Sequence_Length,Hydrophobic_Fraction,Charged_Fraction,Aromatic_Fraction,Polar_Fraction,Small_Residue_Fraction,Glycine_Fraction,Proline_Fraction,Cysteine_Fraction,Basic_Fraction,Acidic_Fraction
0,PE_PROT_001,MQPPPFRGILTPLFPLSSSPPVGSLSRPGRRGVLTRLVAVVALVLG...,True,519,519.0,0.398844,0.136802,0.086705,0.252408,0.437380,0.109827,0.067437,0.015414,0.082852,0.073218
1,PE_PROT_002,MKRLFIAGMIFILFLSLGAVSSSAAGSFTSKTYNGRTYKLYVPSSY...,True,590,590.0,0.389831,0.149153,0.096610,0.296610,0.403390,0.098305,0.049153,0.003390,0.077966,0.084746
2,PE_PROT_003,MLAKQIKKANSRSTLLRKSLLFAAPIILAVSSSSVYALTQVSNFGT...,True,414,414.0,0.384058,0.101449,0.099034,0.306763,0.417874,0.108696,0.057971,0.019324,0.077295,0.045894
3,PE_PROT_004,MTSRPMRSLVIAFLTLVAAAAPALAGAGAWQNNLSLGGFNKVHLYT...,True,635,635.0,0.401575,0.162205,0.080315,0.236220,0.417323,0.129134,0.036220,0.022047,0.058268,0.116535
4,PE_PROT_005,MVRRLWRRIAGWLAACVAILCAFPLHAATAGPGAWSSQQTWAADSV...,True,488,488.0,0.424180,0.081967,0.098361,0.293033,0.465164,0.122951,0.043033,0.016393,0.059426,0.040984


In [55]:
feature_cols_sequence = [
    "Sequence_Length",
    "Hydrophobic_Fraction",
    "Charged_Fraction",
    "Aromatic_Fraction",
    "Polar_Fraction",
    "Small_Residue_Fraction",
    "Glycine_Fraction",
    "Proline_Fraction",
    "Cysteine_Fraction",
    "Basic_Fraction",
    "Acidic_Fraction"
]

sequence_feature_table[feature_cols_sequence].describe().T

,count,mean,std,min,25%,50%,75%,max
Sequence_Length,188.0,342.601064,119.782760,129.000000,262.000000,303.000000,407.250000,914.000000
Hydrophobic_Fraction,188.0,0.414169,0.030310,0.333333,0.393103,0.412132,0.433927,0.488136
Charged_Fraction,188.0,0.163884,0.042355,0.070796,0.139509,0.169295,0.189528,0.301075
Aromatic_Fraction,188.0,0.086546,0.016406,0.049180,0.075123,0.087786,0.098300,0.148014
Polar_Fraction,188.0,0.231194,0.054438,0.112000,0.190019,0.237213,0.269461,0.353349
Small_Residue_Fraction,188.0,0.363596,0.058762,0.188172,0.325355,0.365604,0.401989,0.491667
Glycine_Fraction,188.0,0.096311,0.019960,0.037634,0.081181,0.097320,0.109167,0.145570
Proline_Fraction,188.0,0.059508,0.015580,0.023256,0.047947,0.060852,0.071664,0.101877
Cysteine_Fraction,188.0,0.011842,0.007628,0.000000,0.006661,0.010221,0.017391,0.036408
Basic_Fraction,188.0,0.100874,0.027475,0.044248,0.082155,0.100343,0.119015,0.231183


In [56]:
sequence_corr = (
    sequence_feature_table[feature_cols_sequence]
    .corr(method="spearman")
)

sequence_corr.round(3)

,Sequence_Length,Hydrophobic_Fraction,Charged_Fraction,Aromatic_Fraction,Polar_Fraction,Small_Residue_Fraction,Glycine_Fraction,Proline_Fraction,Cysteine_Fraction,Basic_Fraction,Acidic_Fraction
Sequence_Length,1.000,0.012,-0.099,0.261,0.135,0.152,0.090,-0.112,0.123,-0.228,-0.028
Hydrophobic_Fraction,0.012,1.000,0.166,-0.200,-0.525,-0.262,-0.144,-0.060,-0.216,0.132,0.106
Charged_Fraction,-0.099,0.166,1.000,-0.066,-0.800,-0.770,-0.420,0.238,-0.502,0.814,0.895
Aromatic_Fraction,0.261,-0.200,-0.066,1.000,0.197,-0.147,-0.134,-0.041,0.150,-0.095,0.003
Polar_Fraction,0.135,-0.525,-0.800,0.197,1.000,0.699,0.184,-0.331,0.421,-0.711,-0.719
Small_Residue_Fraction,0.152,-0.262,-0.770,-0.147,0.699,1.000,0.581,-0.340,0.358,-0.699,-0.719
Glycine_Fraction,0.090,-0.144,-0.420,-0.134,0.184,0.581,1.000,-0.384,0.262,-0.444,-0.368
Proline_Fraction,-0.112,-0.060,0.238,-0.041,-0.331,-0.340,-0.384,1.000,-0.235,0.294,0.216
Cysteine_Fraction,0.123,-0.216,-0.502,0.150,0.421,0.358,0.262,-0.235,1.000,-0.529,-0.439
Basic_Fraction,-0.228,0.132,0.814,-0.095,-0.711,-0.699,-0.444,0.294,-0.529,1.000,0.576


In [58]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import pandas as pd

def additional_sequence_features(seq):
    clean_seq = seq.replace("X", "")
    analysis = ProteinAnalysis(clean_seq)
    length = len(clean_seq)

    A = clean_seq.count("A") / length * 100
    V = clean_seq.count("V") / length * 100
    I = clean_seq.count("I") / length * 100
    L = clean_seq.count("L") / length * 100

    aliphatic_index = A + 2.9 * V + 3.9 * (I + L)

    return pd.Series({
        "Theoretical_pI": analysis.isoelectric_point(),
        "Instability_Index": analysis.instability_index(),
        "Aliphatic_Index": aliphatic_index,
        "GRAVY": analysis.gravy()
    })

additional_features = (
    protein_sequences["SEQUENCE_NORMALIZED"]
    .apply(additional_sequence_features)
)

additional_features.describe().T

,count,mean,std,min,25%,50%,75%,max
Theoretical_pI,188.0,6.851744,1.604928,4.179110,5.567715,6.400917,8.236255,11.409115
Instability_Index,188.0,34.366990,8.165472,17.725221,28.761341,34.816594,39.205049,69.411013
Aliphatic_Index,188.0,80.155839,10.989087,56.706444,72.053180,79.657082,88.087951,110.149254
GRAVY,188.0,-0.131137,0.162149,-0.549762,-0.234183,-0.132463,-0.005718,0.239352


In [59]:
all_sequence_features = pd.concat(
    [
        sequence_feature_table[feature_cols_sequence],
        additional_features
    ],
    axis=1
)

all_sequence_corr = all_sequence_features.corr(method="spearman")

all_sequence_corr.round(3)

,Sequence_Length,Hydrophobic_Fraction,Charged_Fraction,Aromatic_Fraction,Polar_Fraction,Small_Residue_Fraction,Glycine_Fraction,Proline_Fraction,Cysteine_Fraction,Basic_Fraction,Acidic_Fraction,Theoretical_pI,Instability_Index,Aliphatic_Index,GRAVY
Sequence_Length,1.000,0.012,-0.099,0.261,0.135,0.152,0.090,-0.112,0.123,-0.228,-0.028,-0.150,-0.051,-0.284,-0.083
Hydrophobic_Fraction,0.012,1.000,0.166,-0.200,-0.525,-0.262,-0.144,-0.060,-0.216,0.132,0.106,-0.008,0.050,0.717,0.710
Charged_Fraction,-0.099,0.166,1.000,-0.066,-0.800,-0.770,-0.420,0.238,-0.502,0.814,0.895,-0.262,0.386,0.453,-0.313
Aromatic_Fraction,0.261,-0.200,-0.066,1.000,0.197,-0.147,-0.134,-0.041,0.150,-0.095,0.003,-0.125,0.015,-0.467,-0.433
Polar_Fraction,0.135,-0.525,-0.800,0.197,1.000,0.699,0.184,-0.331,0.421,-0.711,-0.719,0.204,-0.345,-0.641,-0.086
Small_Residue_Fraction,0.152,-0.262,-0.770,-0.147,0.699,1.000,0.581,-0.340,0.358,-0.699,-0.719,0.242,-0.461,-0.561,0.168
Glycine_Fraction,0.090,-0.144,-0.420,-0.134,0.184,0.581,1.000,-0.384,0.262,-0.444,-0.368,0.025,-0.478,-0.303,0.205
Proline_Fraction,-0.112,-0.060,0.238,-0.041,-0.331,-0.340,-0.384,1.000,-0.235,0.294,0.216,-0.051,0.572,0.067,-0.172
Cysteine_Fraction,0.123,-0.216,-0.502,0.150,0.421,0.358,0.262,-0.235,1.000,-0.529,-0.439,0.079,-0.236,-0.365,0.136
Basic_Fraction,-0.228,0.132,0.814,-0.095,-0.711,-0.699,-0.444,0.294,-0.529,1.000,0.576,0.161,0.428,0.403,-0.322


In [60]:
corr_pairs = (
    all_sequence_corr
    .where(
        np.triu(
            np.ones(all_sequence_corr.shape),
            k=1
        ).astype(bool)
    )
    .stack()
    .reset_index()
)

corr_pairs.columns = ["Feature_1", "Feature_2", "Correlation"]

corr_pairs[
    corr_pairs["Correlation"].abs() >= 0.75
].sort_values(
    "Correlation",
    key=lambda x: x.abs(),
    ascending=False
)

,Feature_1,Feature_2,Correlation
40,Charged_Fraction,Acidic_Fraction,0.894509
39,Charged_Fraction,Basic_Fraction,0.814395
34,Charged_Fraction,Polar_Fraction,-0.800168
35,Charged_Fraction,Small_Residue_Fraction,-0.769763


In [61]:
all_sequence_features = pd.concat(
    [
        sequence_feature_table[feature_cols_sequence],
        additional_features
    ],
    axis=1
)

all_sequence_corr = all_sequence_features.corr(method="spearman")

all_sequence_corr.round(3)

,Sequence_Length,Hydrophobic_Fraction,Charged_Fraction,Aromatic_Fraction,Polar_Fraction,Small_Residue_Fraction,Glycine_Fraction,Proline_Fraction,Cysteine_Fraction,Basic_Fraction,Acidic_Fraction,Theoretical_pI,Instability_Index,Aliphatic_Index,GRAVY
Sequence_Length,1.000,0.012,-0.099,0.261,0.135,0.152,0.090,-0.112,0.123,-0.228,-0.028,-0.150,-0.051,-0.284,-0.083
Hydrophobic_Fraction,0.012,1.000,0.166,-0.200,-0.525,-0.262,-0.144,-0.060,-0.216,0.132,0.106,-0.008,0.050,0.717,0.710
Charged_Fraction,-0.099,0.166,1.000,-0.066,-0.800,-0.770,-0.420,0.238,-0.502,0.814,0.895,-0.262,0.386,0.453,-0.313
Aromatic_Fraction,0.261,-0.200,-0.066,1.000,0.197,-0.147,-0.134,-0.041,0.150,-0.095,0.003,-0.125,0.015,-0.467,-0.433
Polar_Fraction,0.135,-0.525,-0.800,0.197,1.000,0.699,0.184,-0.331,0.421,-0.711,-0.719,0.204,-0.345,-0.641,-0.086
Small_Residue_Fraction,0.152,-0.262,-0.770,-0.147,0.699,1.000,0.581,-0.340,0.358,-0.699,-0.719,0.242,-0.461,-0.561,0.168
Glycine_Fraction,0.090,-0.144,-0.420,-0.134,0.184,0.581,1.000,-0.384,0.262,-0.444,-0.368,0.025,-0.478,-0.303,0.205
Proline_Fraction,-0.112,-0.060,0.238,-0.041,-0.331,-0.340,-0.384,1.000,-0.235,0.294,0.216,-0.051,0.572,0.067,-0.172
Cysteine_Fraction,0.123,-0.216,-0.502,0.150,0.421,0.358,0.262,-0.235,1.000,-0.529,-0.439,0.079,-0.236,-0.365,0.136
Basic_Fraction,-0.228,0.132,0.814,-0.095,-0.711,-0.699,-0.444,0.294,-0.529,1.000,0.576,0.161,0.428,0.403,-0.322


In [62]:
corr_pairs = (
    all_sequence_corr
    .where(
        np.triu(
            np.ones(all_sequence_corr.shape),
            k=1
        ).astype(bool)
    )
    .stack()
    .reset_index()
)

corr_pairs.columns = ["Feature_1", "Feature_2", "Correlation"]

corr_pairs[
    corr_pairs["Correlation"].abs() >= 0.75
].sort_values(
    "Correlation",
    key=lambda x: x.abs(),
    ascending=False
)

,Feature_1,Feature_2,Correlation
40,Charged_Fraction,Acidic_Fraction,0.894509
39,Charged_Fraction,Basic_Fraction,0.814395
34,Charged_Fraction,Polar_Fraction,-0.800168
35,Charged_Fraction,Small_Residue_Fraction,-0.769763


In [63]:
additional_features = additional_features.copy()
additional_features["protein_id"] = protein_sequences["protein_id"].values

model1A_features = (
    model1_features_1A[
        ["protein_id", "Type"]
    ]
    .merge(
        sequence_feature_table[
            ["protein_id", "Sequence_Length",
             "Aromatic_Fraction",
             "Proline_Fraction",
             "Cysteine_Fraction"]
        ],
        on="protein_id",
        how="inner"
    )
    .merge(
        additional_features[
            ["protein_id", "Theoretical_pI",
             "Instability_Index", "GRAVY"]
        ],
        on="protein_id",
        how="inner"
    )
)

model1A_features.shape

(188, 9)

In [64]:
model1A_features.isna().sum()

protein_id           0
Type                 0
Sequence_Length      0
Aromatic_Fraction    0
Proline_Fraction     0
Cysteine_Fraction    0
Theoretical_pI       0
Instability_Index    0
GRAVY                0
dtype: int64

Sequence Similarity CLustering 

In [65]:
import shutil

print("CD-HIT:", shutil.which("cd-hit"))
print("MMseqs2:", shutil.which("mmseqs"))

CD-HIT: None
MMseqs2: None


In [66]:
from Bio.Align import PairwiseAligner
from itertools import combinations

sequences = dict(
    zip(
        protein_sequences["protein_id"],
        protein_sequences["SEQUENCE_NORMALIZED"]
    )
)

protein_ids = list(sequences.keys())

aligner = PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 1
aligner.mismatch_score = 0
aligner.open_gap_score = -1
aligner.extend_gap_score = -0.1

candidate_pairs = []

for id1, id2 in combinations(protein_ids, 2):
    seq1 = sequences[id1].replace("X", "")
    seq2 = sequences[id2].replace("X", "")
    
    shorter = min(len(seq1), len(seq2))
    longer = max(len(seq1), len(seq2))
    
    if shorter / longer < 0.80:
        continue
    
    candidate_pairs.append((id1, id2))

print("Total possible pairs:", len(protein_ids) * (len(protein_ids) - 1) // 2)
print("Candidate pairs after length filtering:", len(candidate_pairs))

Total possible pairs: 17578
Candidate pairs after length filtering: 6913


In [67]:
similar_pairs = []

for id1, id2 in candidate_pairs:
    seq1 = sequences[id1].replace("X", "")
    seq2 = sequences[id2].replace("X", "")
    
    alignment = aligner.align(seq1, seq2)[0]
    
    aligned_seq1, aligned_seq2 = alignment
    
    matches = sum(
        a == b
        for a, b in zip(aligned_seq1, aligned_seq2)
        if a != "-" and b != "-"
    )
    
    aligned_positions = sum(
        a != "-" and b != "-"
        for a, b in zip(aligned_seq1, aligned_seq2)
    )
    
    identity = matches / aligned_positions if aligned_positions else 0
    coverage = aligned_positions / min(len(seq1), len(seq2))
    
    if identity >= 0.95 and coverage >= 0.80:
        similar_pairs.append({
            "Protein_1": id1,
            "Protein_2": id2,
            "Identity": identity,
            "Coverage": coverage,
            "Length_1": len(seq1),
            "Length_2": len(seq2)
        })

similar_pairs = pd.DataFrame(similar_pairs)

print("Similar pairs:", len(similar_pairs))
similar_pairs.sort_values(
    ["Identity", "Coverage"],
    ascending=False
).head(20)

Similar pairs: 61


,Protein_1,Protein_2,Identity,Coverage,Length_1,Length_2
3,PE_PROT_014,PE_PROT_090,1.000000,1.000000,301,261
6,PE_PROT_014,PE_PROT_104,1.000000,1.000000,301,262
9,PE_PROT_023,PE_PROT_147,1.000000,1.000000,239,238
14,PE_PROT_065,PE_PROT_172,1.000000,1.000000,213,197
15,PE_PROT_071,PE_PROT_082,1.000000,1.000000,300,240
41,PE_PROT_090,PE_PROT_104,1.000000,1.000000,261,262
44,PE_PROT_093,PE_PROT_129,1.000000,0.972414,308,290
35,PE_PROT_083,PE_PROT_142,1.000000,0.970370,296,270
12,PE_PROT_039,PE_PROT_042,0.996466,1.000000,283,283
31,PE_PROT_081,PE_PROT_140,0.996183,1.000000,262,270


In [68]:
similar_ids = set()

for _, row in similar_pairs.iterrows():
    similar_ids.add(row["Protein_1"])
    similar_ids.add(row["Protein_2"])

adjacency = {pid: set() for pid in similar_ids}

for _, row in similar_pairs.iterrows():
    p1 = row["Protein_1"]
    p2 = row["Protein_2"]
    adjacency[p1].add(p2)
    adjacency[p2].add(p1)

clusters = []
visited = set()

for pid in adjacency:
    if pid in visited:
        continue
    
    stack = [pid]
    cluster = []
    
    while stack:
        current = stack.pop()
        
        if current in visited:
            continue
        
        visited.add(current)
        cluster.append(current)
        
        for neighbor in adjacency[current]:
            if neighbor not in visited:
                stack.append(neighbor)
    
    clusters.append(cluster)

cluster_records = []

for i, cluster in enumerate(clusters, start=1):
    for pid in cluster:
        cluster_records.append({
            "protein_id": pid,
            "similarity_cluster": f"SIM_{i:03d}"
        })

similarity_clusters = pd.DataFrame(cluster_records)

print("Number of similarity clusters:", len(clusters))
print("Proteins in similarity clusters:", len(similar_ids))
print("Singleton proteins:", 188 - len(similar_ids))

Number of similarity clusters: 18
Proteins in similarity clusters: 50
Singleton proteins: 138


In [69]:
cluster_sizes = (
    similarity_clusters
    .groupby("similarity_cluster")
    .size()
    .sort_values(ascending=False)
)

cluster_sizes

similarity_cluster
SIM_002    11
SIM_004     4
SIM_013     4
SIM_010     3
SIM_001     2
SIM_005     2
SIM_006     2
SIM_007     2
SIM_008     2
SIM_003     2
SIM_009     2
SIM_011     2
SIM_012     2
SIM_014     2
SIM_015     2
SIM_016     2
SIM_017     2
SIM_018     2
dtype: int64

In [70]:
cluster_label_table = (
    similarity_clusters
    .merge(
        target_matrix.reset_index(),
        on="protein_id",
        how="left"
    )
)

cluster_label_summary = (
    cluster_label_table
    .groupby("similarity_cluster")[target_polymers]
    .sum()
)

cluster_label_summary

,PET,PHB,PCL,PBSA,PBAT,PLA,PHA,Nylon
similarity_cluster,,,,,,,,
SIM_001,0,0,2,0,0,0,0,0
SIM_002,11,0,0,0,1,0,0,0
SIM_003,0,0,0,0,0,0,0,2
SIM_004,3,0,2,0,1,0,0,0
SIM_005,0,2,0,0,0,0,0,0
SIM_006,2,0,1,0,0,0,0,0
SIM_007,0,1,1,0,0,1,0,0
SIM_008,2,0,1,0,0,0,0,0
SIM_009,1,0,1,0,0,0,0,0


In [72]:
mixed_clusters = cluster_label_summary[
    (cluster_label_summary > 0).sum(axis=1) > 1
]

print("Clusters containing multiple polymer labels:", len(mixed_clusters))

Clusters containing multiple polymer labels: 10


In [73]:
cluster_members = (
    similarity_clusters
    .groupby("similarity_cluster")["protein_id"]
    .apply(list)
)

for cluster_id, members in cluster_members.items():
    print(cluster_id, "->", members)

SIM_001 -> ['PE_PROT_164', 'PE_PROT_165']
SIM_002 -> ['PE_PROT_095', 'PE_PROT_096', 'PE_PROT_014', 'PE_PROT_090', 'PE_PROT_140', 'PE_PROT_141', 'PE_PROT_081', 'PE_PROT_145', 'PE_PROT_084', 'PE_PROT_072', 'PE_PROT_104']
SIM_003 -> ['PE_PROT_080', 'PE_PROT_076']
SIM_004 -> ['PE_PROT_142', 'PE_PROT_071', 'PE_PROT_082', 'PE_PROT_083']
SIM_005 -> ['PE_PROT_005', 'PE_PROT_030']
SIM_006 -> ['PE_PROT_109', 'PE_PROT_106']
SIM_007 -> ['PE_PROT_147', 'PE_PROT_023']
SIM_008 -> ['PE_PROT_085', 'PE_PROT_091']
SIM_009 -> ['PE_PROT_102', 'PE_PROT_089']
SIM_010 -> ['PE_PROT_112', 'PE_PROT_110', 'PE_PROT_111']
SIM_011 -> ['PE_PROT_042', 'PE_PROT_039']
SIM_012 -> ['PE_PROT_056', 'PE_PROT_057']
SIM_013 -> ['PE_PROT_074', 'PE_PROT_075', 'PE_PROT_073', 'PE_PROT_078']
SIM_014 -> ['PE_PROT_038', 'PE_PROT_041']
SIM_015 -> ['PE_PROT_002', 'PE_PROT_148']
SIM_016 -> ['PE_PROT_035', 'PE_PROT_029']
SIM_017 -> ['PE_PROT_065', 'PE_PROT_172']
SIM_018 -> ['PE_PROT_093', 'PE_PROT_129']


In [74]:
from itertools import combinations
from Bio.Align import PairwiseAligner
import pandas as pd

aligner = PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 1
aligner.mismatch_score = 0
aligner.open_gap_score = -1
aligner.extend_gap_score = -0.1

exhaustive_similar_pairs = []

for id1, id2 in combinations(protein_ids, 2):
    seq1 = sequences[id1].replace("X", "")
    seq2 = sequences[id2].replace("X", "")
    
    alignment = aligner.align(seq1, seq2)[0]
    aligned_seq1, aligned_seq2 = alignment
    
    matches = sum(
        a == b
        for a, b in zip(aligned_seq1, aligned_seq2)
        if a != "-" and b != "-"
    )
    
    aligned_positions = sum(
        a != "-" and b != "-"
        for a, b in zip(aligned_seq1, aligned_seq2)
    )
    
    identity = matches / aligned_positions if aligned_positions else 0
    coverage = aligned_positions / min(len(seq1), len(seq2))
    
    if identity >= 0.95 and coverage >= 0.80:
        exhaustive_similar_pairs.append({
            "Protein_1": id1,
            "Protein_2": id2,
            "Identity": identity,
            "Coverage": coverage,
            "Length_1": len(seq1),
            "Length_2": len(seq2)
        })

exhaustive_similar_pairs = pd.DataFrame(exhaustive_similar_pairs)

print("Total pairs checked:", len(protein_ids) * (len(protein_ids) - 1) // 2)
print("Similar pairs:", len(exhaustive_similar_pairs))

exhaustive_similar_pairs = exhaustive_similar_pairs.sort_values(
    ["Identity", "Coverage"],
    ascending=False
).reset_index(drop=True)

exhaustive_similar_pairs.head(20)

Total pairs checked: 17578
Similar pairs: 62


,Protein_1,Protein_2,Identity,Coverage,Length_1,Length_2
0,PE_PROT_014,PE_PROT_090,1.000000,1.000000,301,261
1,PE_PROT_014,PE_PROT_104,1.000000,1.000000,301,262
2,PE_PROT_023,PE_PROT_147,1.000000,1.000000,239,238
3,PE_PROT_065,PE_PROT_172,1.000000,1.000000,213,197
4,PE_PROT_071,PE_PROT_082,1.000000,1.000000,300,240
5,PE_PROT_090,PE_PROT_104,1.000000,1.000000,261,262
6,PE_PROT_093,PE_PROT_129,1.000000,0.972414,308,290
7,PE_PROT_083,PE_PROT_142,1.000000,0.970370,296,270
8,PE_PROT_039,PE_PROT_042,0.996466,1.000000,283,283
9,PE_PROT_081,PE_PROT_140,0.996183,1.000000,262,270


In [75]:
similar_ids = set()

for _, row in exhaustive_similar_pairs.iterrows():
    similar_ids.add(row["Protein_1"])
    similar_ids.add(row["Protein_2"])

adjacency = {pid: set() for pid in similar_ids}

for _, row in exhaustive_similar_pairs.iterrows():
    p1 = row["Protein_1"]
    p2 = row["Protein_2"]
    adjacency[p1].add(p2)
    adjacency[p2].add(p1)

clusters = []
visited = set()

for pid in adjacency:
    if pid in visited:
        continue
    
    stack = [pid]
    cluster = []
    
    while stack:
        current = stack.pop()
        
        if current in visited:
            continue
        
        visited.add(current)
        cluster.append(current)
        
        for neighbor in adjacency[current]:
            if neighbor not in visited:
                stack.append(neighbor)
    
    clusters.append(cluster)

similarity_clusters_final = []

for i, cluster in enumerate(clusters, start=1):
    for pid in cluster:
        similarity_clusters_final.append({
            "protein_id": pid,
            "similarity_cluster": f"SIM_{i:03d}"
        })

similarity_clusters_final = pd.DataFrame(similarity_clusters_final)

print("Number of clusters:", len(clusters))
print("Proteins in non-singleton clusters:", len(similar_ids))
print("Singleton proteins:", len(protein_ids) - len(similar_ids))

Number of clusters: 19
Proteins in non-singleton clusters: 52
Singleton proteins: 136


In [76]:
from collections import defaultdict

protein_to_group = {}

for i, cluster in enumerate(clusters, start=1):
    group_name = f"SIM_{i:03d}"
    for protein_id in cluster:
        protein_to_group[protein_id] = group_name

singleton_counter = 1

for protein_id in protein_ids:
    if protein_id not in protein_to_group:
        protein_to_group[protein_id] = f"SINGLE_{singleton_counter:03d}"
        singleton_counter += 1

protein_groups = pd.DataFrame({
    "protein_id": protein_ids,
    "group": [protein_to_group[p] for p in protein_ids]
})

print("Total groups:", protein_groups["group"].nunique())
print("Total proteins:", len(protein_groups))
print("Non-singleton groups:", sum(
    protein_groups["group"].value_counts() > 1
))
print("Singleton groups:", sum(
    protein_groups["group"].value_counts() == 1
))

Total groups: 155
Total proteins: 188
Non-singleton groups: 19
Singleton groups: 136


In [77]:
target_polymers = [
    "PET",
    "PHB",
    "PCL",
    "PBSA",
    "PBAT",
    "PLA",
    "PHA",
    "Nylon"
]

group_table = (
    protein_groups
    .merge(
        target_matrix.reset_index(),
        on="protein_id",
        how="left"
    )
    .groupby("group")
    .agg(
        group_size=("protein_id", "size"),
        **{
            polymer: (polymer, "sum")
            for polymer in target_polymers
        }
    )
    .reset_index()
)

overall_counts = target_matrix[target_polymers].sum()

n_total = len(target_matrix)
desired_test_size = round(n_total * 0.20)

rng = np.random.default_rng(42)

best_score = float("inf")
best_test_groups = None
best_test_counts = None

for _ in range(200000):

    selected = rng.random(len(group_table)) < 0.20

    test_groups = group_table.loc[selected]

    test_size = test_groups["group_size"].sum()

    if test_size < 35 or test_size > 41:
        continue

    test_counts = test_groups[target_polymers].sum()
    train_counts = overall_counts - test_counts

    if (test_counts == 0).any():
        continue

    if (train_counts == 0).any():
        continue

    proportion_error = (
        (test_counts / overall_counts - 0.20) ** 2
    ).sum()

    size_error = ((test_size - desired_test_size) / n_total) ** 2

    score = proportion_error + 5 * size_error

    if score < best_score:
        best_score = score
        best_test_groups = test_groups["group"].tolist()
        best_test_counts = test_counts.copy()

print("Best score:", best_score)
print("Test proteins:", int(best_test_counts.sum()))

Best score: 0.005234700353891781
Test proteins: 48


In [78]:
test_group_set = set(best_test_groups)

test_proteins = protein_groups.loc[
    protein_groups["group"].isin(test_group_set),
    "protein_id"
].tolist()

train_proteins = protein_groups.loc[
    ~protein_groups["group"].isin(test_group_set),
    "protein_id"
].tolist()

print("Train proteins:", len(train_proteins))
print("Test proteins:", len(test_proteins))

Train proteins: 149
Test proteins: 39


In [ ]:
train_matrix = target_matrix.loc[train_proteins]
test_matrix = target_matrix.loc[test_proteins]

split_summary = pd.DataFrame({
    "Total": target_matrix[target_polymers].sum(),
    "Train": train_matrix[target_polymers].sum(),
    "Test": test_matrix[target_polymers].sum()
})

split_summary["Train_%"] = (
    split_summary["Train"] /
    split_summary["Total"] * 100
)

split_summary["Test_%"] = (
    split_summary["Test"] /
    split_summary["Total"] * 100
)


split_summary

,Total,Train,Test,Train_%,Test_%
PLASTIC,,,,,
PET,71,57,14,80.281690,19.718310
PHB,50,39,11,78.000000,22.000000
PCL,43,34,9,79.069767,20.930233
PBSA,20,16,4,80.000000,20.000000
PBAT,18,14,4,77.777778,22.222222
PLA,12,10,2,83.333333,16.666667
PHA,8,6,2,75.000000,25.000000
Nylon,9,7,2,77.777778,22.222222


In [85]:
train_groups = set(
    protein_groups.loc[
        protein_groups["protein_id"].isin(train_proteins),
        "group"
    ]
)

test_groups = set(
    protein_groups.loc[
        protein_groups["protein_id"].isin(test_proteins),
        "group"
    ]
)

print("Groups in both train and test:", train_groups & test_groups)

Groups in both train and test: set()


In [86]:
assert train_groups.isdisjoint(test_groups)
assert len(set(train_proteins) & set(test_proteins)) == 0

print("No protein overlap.")
print("No similarity-group leakage.")

No protein overlap.
No similarity-group leakage.


In [87]:
train_groups = set(
    protein_groups.loc[
        protein_groups["protein_id"].isin(train_proteins),
        "group"
    ]
)

test_groups = set(
    protein_groups.loc[
        protein_groups["protein_id"].isin(test_proteins),
        "group"
    ]
)

overlap_groups = train_groups & test_groups

print("Groups in both train and test:", overlap_groups)

Groups in both train and test: set()


In [88]:
assert train_groups.isdisjoint(test_groups)
assert set(train_proteins).isdisjoint(set(test_proteins))

print("Split validation passed.")
print("Train proteins:", len(train_proteins))
print("Test proteins:", len(test_proteins))
print("Train groups:", len(train_groups))
print("Test groups:", len(test_groups))

Split validation passed.
Train proteins: 149
Test proteins: 39
Train groups: 120
Test groups: 35


In [89]:
feature_cols_1A = [
    "Type",
    "Sequence_Length",
    "GRAVY",
    "Aromatic_Fraction",
    "Proline_Fraction",
    "Cysteine_Fraction",
    "Theoretical_pI",
    "Instability_Index"
]

X_1A = model1A_features.set_index("protein_id")[feature_cols_1A]

X_1A = pd.get_dummies(
    X_1A,
    columns=["Type"],
    dtype=int
)

X_train = X_1A.loc[train_proteins]
X_test = X_1A.loc[test_proteins]

y_train = target_matrix.loc[train_proteins]
y_test = target_matrix.loc[test_proteins]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (149, 9)
X_test: (39, 9)
y_train: (149, 8)
y_test: (39, 8)


In [90]:
imbalance_1A = pd.DataFrame({
    "Negative": (1 - y_train[target_polymers]).sum(),
    "Positive": y_train[target_polymers].sum()
})

imbalance_1A["scale_pos_weight"] = (
    imbalance_1A["Negative"] /
    imbalance_1A["Positive"]
)

imbalance_1A

,Negative,Positive,scale_pos_weight
PLASTIC,,,
PET,92,57,1.614035
PHB,110,39,2.820513
PCL,115,34,3.382353
PBSA,133,16,8.312500
PBAT,135,14,9.642857
PLA,139,10,13.900000
PHA,143,6,23.833333
Nylon,142,7,20.285714
